# 🧪 Финальная практика: собственный LLM-судья

**Задача:** написать промпт, с помощью которого LLM сравнит чужой ответ с эталоном, выставит **оценку 0–4** и поможет решить, принимать ответ или считать его галлюцинацией.

**Что уже готово:** CSV с 30 парами ответов, подключение к **OpenAI / GigaChat**, отправка запросов, проверка JSON, сохранение результатов, выбор порога и расчёт метрик.

**Что делают участники:** самостоятельно пишут промпт (роль + задача + шкала + формат JSON), проверяют его сначала на **3 примерах**, затем на **всех 30**, выбирают порог и оценивают классификатор.

> **Важно:** это не библиотека RAGAS, а отдельная мини-система **LLM-as-a-Judge** для фактической сверки с эталоном. Эталон в упражнении считаем единственным источником истины, даже если текст модели звучит правдоподобно.

## 1. Договоримся о разметке

- `llm_answer` — проверяемый ответ другой модели (**1-й столбец CSV**).
- `reference_answer` — эталонный ответ (**2-й столбец CSV**).
- `has_hallucination` — правильный класс (**3-й столбец CSV**): `1` = есть ошибка или неподтверждённый факт, `0` = нет.

**Что считать галлюцинацией в этой практике:** любое фактическое утверждение в `llm_answer`, которое **противоречит эталону** или **не подтверждается им**. Отсутствие некоторой информации из эталона не является галлюцинацией, если сказанное в ответе верно. Перефразирование допустимо.

Все 30 примеров синтетические: реальные знания о мире не нужны. В настоящих системах один краткий эталон часто не охватывает всех истинных фактов — этот эксперимент сознательно упрощён.

In [ ]:
%pip -q install "openai>=1.60,<3" "gigachat>=0.1.30" "pandas>=2" "scikit-learn>=1.4" "matplotlib>=3.7" "tqdm>=4.65"

## 2. Загружаем CSV

Скачайте приложенный `final_llm_judge_30_cases.csv` и загрузите его в Colab при выполнении следующей ячейки. Если запускаете Jupyter локально, положите файл рядом с ноутбуком или измените `CSV_PATH`.

In [ ]:
from pathlib import Path
import pandas as pd

CSV_PATH = "final_llm_judge_30_cases.csv"

if not Path(CSV_PATH).exists():
    try:
        from google.colab import files
        print("Выберите final_llm_judge_30_cases.csv")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Файл не загружен")
        CSV_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Поместите CSV в рабочую папку: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
required = {"llm_answer", "reference_answer", "has_hallucination"}
assert required.issubset(df.columns), f"В CSV не хватает колонок: {required - set(df.columns)}"
assert df["has_hallucination"].isin([0, 1]).all(), "Метки должны быть 0 или 1"
assert df[["llm_answer", "reference_answer"]].notna().all().all(), "Есть пустые ответы"
df["case_id"] = range(1, len(df) + 1)
print(f"Загружено {len(df)} примеров, из них с ошибками: {int(df.has_hallucination.sum())}")
display(df[["case_id", "llm_answer", "reference_answer"]].head(5))
# Метки не показываем в этой таблице и НЕ отправляем LLM-судье.

## 3. Настраиваем модель-судью

Поддерживаются **OpenAI** и **GigaChat**. Ключ вводится скрыто. Для GigaChat при необходимости укажите свой `GIGACHAT_SCOPE`.

Модель получает **только два текста** и промпт студента. Колонка `has_hallucination` передаётся только функциям оценки качества.

In [ ]:
from getpass import getpass

PROVIDER = "openai"         # "openai" или "gigachat"
JUDGE_MODEL = "gpt-4.1-mini" # Для GigaChat, например: "GigaChat-2" или "GigaChat-2-Pro" (если доступна)
GIGACHAT_SCOPE = "GIGACHAT_API_PERS"  # При необходимости смените на B2B / CORP
GIGACHAT_VERIFY_SSL = False

API_KEY = getpass(f"API-ключ {PROVIDER}: ")
if not API_KEY.strip():
    raise ValueError("Укажите API-ключ")

if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, timeout=60, max_retries=2)
elif PROVIDER == "gigachat":
    from gigachat import GigaChat
    client = GigaChat(
        credentials=API_KEY,
        scope=GIGACHAT_SCOPE,
        model=JUDGE_MODEL,
        verify_ssl_certs=GIGACHAT_VERIFY_SSL,
        timeout=60,
        max_retries=2,
    )
else:
    raise ValueError("PROVIDER должен быть openai или gigachat")

print(f"Провайдер: {PROVIDER}; модель: {JUDGE_MODEL}")

## 4. ⭐ ВАШЕ ЗАДАНИЕ: напишите промпт

Ориентир — слайд с промптом Fact-Checker: **роль**, **задача**, **шкала 0–4**, **правила сравнения** и **строгий JSON**.

Продумайте: как оценивать неверные числа? Как отличать грубую ошибку от мелкой? Что делать с фактами, которых нет в эталоне? Нужно ли наказывать за неполный, но верный ответ? Чем оценки `3` и `4` отличаются?

В ответе обязательно ожидаются следующие поля:

```json
{
  "accuracy_score": 2,
  "verdict": "Mixed",
  "error_type": "Unsupported information",
  "fact_check_details": "Что не совпало с эталоном",
  "reasoning": "Почему выбрана такая оценка",
  "correction": "Как исправить ответ, если нужно"
}
```

Это **образец структуры**, а не готовый промпт и не правильная оценка конкретного примера. **Оценка 4 — максимально достоверный ответ, 0 — самый проблемный.**

In [ ]:
# ✍️ ИЗМЕНЯТЬ ТОЛЬКО ЭТУ ЯЧЕЙКУ
STUDENT_PROMPT = """
РОЛЬ: [Напишите, кем является модель-судья]

ЗАДАЧА: [Что и на основании чего нужно проверить?]

ШКАЛА (0–4):
0 — [ваши критерии]
1 — [ваши критерии]
2 — [ваши критерии]
3 — [ваши критерии]
4 — [ваши критерии]

ПРАВИЛА: [Что делать с перефразированиями, пропусками и новыми фактами?]

ФОРМАТ: [Опишите строгий JSON с полями
accuracy_score, verdict, error_type, fact_check_details, reasoning, correction]
"""

## 5. Сервисный код (не нужно редактировать)

Один запрос — одна пара ответов. Код **не передаёт судье разметку**. Запросы к API кешируются в пределах текущей сессии: уже проверенные три примера не будут повторно оплачиваться при запуске всех 30 с тем же промптом и моделью.

Если ответ не является корректным JSON со всеми ожидаемыми полями, он помечается как техническая ошибка — не подменяем плохой ответ модели выдуманной оценкой.

In [ ]:
import hashlib
import json
import re
import time
from tqdm.auto import tqdm

REQUIRED_JSON_KEYS = {
    "accuracy_score", "verdict", "error_type",
    "fact_check_details", "reasoning", "correction"
}
CACHE = {}

def check_student_prompt():
    if "[Напишите" in STUDENT_PROMPT or "[ваши критерии]" in STUDENT_PROMPT:
        raise ValueError("Сначала заполните STUDENT_PROMPT в ячейке 4")
    if len(STUDENT_PROMPT.strip()) < 120:
        raise ValueError("Промпт слишком короткий: опишите роль, задачу, шкалу и JSON")

def ask_model(prompt_text):
    if PROVIDER == "openai":
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt_text}],
            temperature=0,
        )
        return response.choices[0].message.content or ""
    # GigaChat Python SDK
    response = client.chat.create(prompt_text)
    return response.messages[0].content[0].text

def parse_judge_json(text):
    clean = text.strip()
    # Убираем возможное обрамление ```json ... ```.
    clean = re.sub(r"^```(?:json)?\s*|\s*```$", "", clean, flags=re.I).strip()
    obj = json.loads(clean)
    if not isinstance(obj, dict):
        raise ValueError("Ожидается JSON-объект")
    missing = REQUIRED_JSON_KEYS - set(obj)
    if missing:
        raise ValueError(f"Нет полей: {sorted(missing)}")
    score = obj["accuracy_score"]
    if type(score) is not int or not 0 <= score <= 4:
        raise ValueError("accuracy_score должен быть целым числом 0..4")
    return obj

def evaluate_one(row):
    check_student_prompt()
    prompt = (STUDENT_PROMPT.strip()
        + "\n\n=== ЭТАЛОННЫЙ ОТВЕТ ===\n" + str(row.reference_answer)
        + "\n\n=== ОТВЕТ LLM ===\n" + str(row.llm_answer)
        + "\n\nВерни результат по заданному формату JSON.")
    fingerprint = hashlib.sha256((PROVIDER + JUDGE_MODEL + prompt).encode()).hexdigest()
    if fingerprint not in CACHE:
        try:
            raw = ask_model(prompt)
            parsed = parse_judge_json(raw)
            record = {
                "case_id": int(row.case_id), "accuracy_score": parsed["accuracy_score"],
                "verdict": parsed["verdict"], "error_type": parsed["error_type"],
                "fact_check_details": parsed["fact_check_details"],
                "reasoning": parsed["reasoning"], "correction": parsed["correction"],
                "raw_answer": raw, "api_error": None,
            }
        except Exception as e:
            record = {"case_id": int(row.case_id), "accuracy_score": None,
                      "verdict": None, "error_type": None,
                      "fact_check_details": None, "reasoning": None,
                      "correction": None, "raw_answer": None,
                      "api_error": f"{type(e).__name__}: {e}"}
        CACHE[fingerprint] = record
    return CACHE[fingerprint].copy()

def evaluate_many(frame):
    results = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc="LLM-судья"):
        results.append(evaluate_one(row))
    return pd.DataFrame(results)

## 6. Первые три примера — быстрая отладка

Посмотрите, совпадает ли структура JSON с ожиданиями, насколько разумны оценки и объяснения. Не меняйте метки в исходном CSV. Если LLM выдала невалидный JSON, уточните требование к формату в своём промпте и попробуйте снова.

In [ ]:
# Стабильные первые три примера: один корректный, один с явной ошибкой и один неполный, но корректный.
preview_df = df.iloc[:3].copy()
preview_scores = evaluate_many(preview_df)
preview_view = preview_df.merge(preview_scores, on="case_id", how="left")
display(preview_view[["case_id", "llm_answer", "reference_answer",
                      "accuracy_score", "reasoning", "api_error"]])

if preview_scores["accuracy_score"].notna().sum() == 0:
    print("⚠️ Все 3 вызова неуспешны. Проверьте API-ключ, название модели и JSON-промпт.")
else:
    print("Успешных оценок:", int(preview_scores["accuracy_score"].notna().sum()), "/ 3")

## 7. Запускаем судью на всех 30 примерах

**Этот шаг может стоить денег:** по одному вызову API на каждый ещё не обработанный пример. Если промпт не менялся, уже оценённые 3 примера будут взяты из кеша.

In [ ]:
all_scores = evaluate_many(df)
results = df.merge(all_scores, on="case_id", how="left", validate="one_to_one")
print(f"Оценок получено: {results.accuracy_score.notna().sum()} из {len(results)}")
display(results[["case_id", "llm_answer", "accuracy_score", "reasoning", "api_error"]].head(10))
if results.accuracy_score.notna().sum() == 0:
    raise RuntimeError("Нет ни одной корректной оценки. Исправьте промпт / подключение.")

## 8. Подбираем порог по DEV, а не по финальному тесту

Наша шкала: **чем выше оценка, тем надёжнее ответ**. Если `accuracy_score >= THRESHOLD` — принимаем ответ (`0` = нет галлюцинации); иначе отклоняем (`1` = галлюцинация).

Чтобы порог не был подобран по тем же примерам, на которых отчитываемся о качестве, разделим **готовые оценки** на `DEV` (12 строк) и `TEST` (18 строк) с сохранением доли ошибок. Выбираем порог **только на DEV**, потом один раз оцениваем TEST.

Для учебного исследования можно посмотреть результаты на всех 30, но тестовая выборка честнее показывает обобщение.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, balanced_accuracy_score,
)

DEV_IDS, TEST_IDS = train_test_split(
    df["case_id"], test_size=0.6, random_state=42,
    stratify=df["has_hallucination"]
)
dev = results[results.case_id.isin(DEV_IDS)].copy()
test = results[results.case_id.isin(TEST_IDS)].copy()

# 1 означает ГАЛЛЮЦИНАЦИЮ — именно её precision/recall мы считаем.
def quality_at_threshold(frame, threshold):
    valid = frame.dropna(subset=["accuracy_score"]).copy()
    if valid.empty:
        return {"n": 0, "coverage": 0.0, "accuracy": float("nan"),
                "precision": float("nan"), "recall": float("nan"),
                "f1": float("nan"), "balanced_accuracy": float("nan")}
    pred = (valid.accuracy_score < threshold).astype(int)
    true = valid.has_hallucination.astype(int)
    return {
        "n": len(valid), "coverage": len(valid) / len(frame),
        "accuracy": accuracy_score(true, pred),
        "precision": precision_score(true, pred, zero_division=0),
        "recall": recall_score(true, pred, zero_division=0),
        "f1": f1_score(true, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
    }

thresholds = []
for t in [1, 2, 3, 4]:
    thresholds.append({"threshold": t, **quality_at_threshold(dev, t)})
threshold_table = pd.DataFrame(thresholds)
print("Метрики только на DEV (12 примеров):")
display(threshold_table.round(3))

## 9. ⭐ Ваш выбор: порог принятия ответа

**Выберите порог сами.** Например, `THRESHOLD = 3` означает, что оценки 3 и 4 — «OK», оценки 0–2 — «галлюцинация». Можно выбрать 1, 2, 3 или 4; объясните свой выбор через баланс **Precision / Recall**.

Если **опаснее пропустить галлюцинацию**, нужен высокий Recall. Если **опаснее зря забраковать правильный ответ**, нужен высокий Precision. Не подбирайте порог по TEST.

In [ ]:
# ✍️ ИЗМЕНИТЬ: выберите порог по таблице DEV выше
THRESHOLD = 3
assert THRESHOLD in [1, 2, 3, 4]

## 10. Финальная оценка качества классификатора

Ниже метрики на **TEST** и **всех 30 примерах**. Отсутствующий или невалидный JSON **не считается правильной классификацией**: его доля отдельно показана как `coverage`. Другие метрики вычисляются **только по строкам, где ответ судьи удалось разобрать**. В реальном проекте технические ошибки тоже нужно контролировать.

In [ ]:
quality_test = quality_at_threshold(test, THRESHOLD)
quality_all = quality_at_threshold(results, THRESHOLD)
quality_comparison = pd.DataFrame([
    {"sample": "TEST (18)", **quality_test},
    {"sample": "ALL (30)", **quality_all},
])
display(quality_comparison.round(3))

valid_test = test.dropna(subset=["accuracy_score"]).copy()
if len(valid_test):
    y_true = valid_test["has_hallucination"].astype(int)
    y_pred = (valid_test["accuracy_score"] < THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"Confusion matrix для TEST: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print("TP — поймали галлюцинацию; FN — пропустили галлюцинацию")
    print("FP — ошибочно забраковали верный ответ; TN — приняли верный ответ")
    mistakes = valid_test.loc[y_true.to_numpy() != y_pred.to_numpy()].copy()
    mistakes["predicted_hallucination"] = (mistakes.accuracy_score < THRESHOLD).astype(int)
    display(mistakes[["case_id", "llm_answer", "reference_answer", "has_hallucination",
                     "accuracy_score", "predicted_hallucination", "reasoning"]])
else:
    print("На TEST нет распознанных оценок — проверьте API и JSON.")

## 11. Визуализация и выгрузка результатов

Сравните, как меняется полнота обнаружения галлюцинаций и точность с изменением порога. CSV с предсказаниями можно разобрать вместе с группой.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
for metric in ["precision", "recall", "f1"]:
    ax.plot(threshold_table["threshold"], threshold_table[metric], marker="o", label=metric)
ax.set(xlabel="Порог (score ниже порога = галлюцинация)", ylabel="Значение метрики", ylim=(-0.04, 1.04))
ax.set_xticks([1, 2, 3, 4])
ax.legend()
ax.grid(alpha=0.25)
plt.show()

results["predicted_hallucination"] = pd.NA
mask = results.accuracy_score.notna()
results.loc[mask, "predicted_hallucination"] = (
    results.loc[mask, "accuracy_score"] < THRESHOLD
).astype(int)
results["sample"] = results.case_id.map(lambda x: "DEV" if x in set(DEV_IDS) else "TEST")
RESULT_CSV = "llm_judge_predictions.csv"
results.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")
print("Сохранено:", RESULT_CSV)
try:
    from google.colab import files
    files.download(RESULT_CSV)
except ImportError:
    pass

## 12. Что означают метрики

Считаем **положительным классом наличие галлюцинации** (`1`).

| Метрика | Смысл | На что обращать внимание |
|---|---|---|
| **Accuracy** | `(TP + TN) / (TP + TN + FP + FN)` | Доля всех верных решений; может обманывать на несбалансированных данных. |
| **Precision** | `TP / (TP + FP)` | Среди найденных галлюцинаций сколько действительно ошибочные? |
| **Recall / Полнота** | `TP / (TP + FN)` | Какую долю реальных галлюцинаций мы поймали? |
| **F1** | `2 × Precision × Recall / (Precision + Recall)` | Баланс Precision и Recall. |
| **Balanced accuracy** | Среднее Recall для класса 0 и класса 1 | Удобно, когда классы не сбалансированы. |
| **Coverage** | Доля примеров с корректным JSON и оценкой 0–4 | Показывает устойчивость промпта и API. |

**TP:** галлюцинация была — судья её обнаружил. **FN:** галлюцинация была — судья пропустил. **FP:** ответ правильный — судья ошибочно отклонил. **TN:** правильный ответ принят.

> **Важно:** высокая оценка самого LLM-судьи (`accuracy_score=4`) — это *не* метрика Accuracy. Оценку 0–4 ставит модель; Accuracy вычисляется позже **по размеченному CSV**.

### Вопросы для обсуждения

1. Где судья ошибался чаще: на числах, на неполных ответах или на дополнительных деталях?
2. При каком пороге меньше **пропущенных галлюцинаций**? Сколько при этом появилось ложных срабатываний?
3. Почему судья может ошибаться, даже если эталон выглядит однозначным?
4. Как изменится F1, если поменять описание роли, задачи, критериев оценки или требование к JSON?